In [1]:
# --- Configuración de entorno ---

# Añade el directorio raíz al path para que Python encuentre tus módulos
import sys, os
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "../..")))  # Sube dos niveles hasta /src

# Importa configuraciones y librerías globales
from config import *
from utils import *

# Carga de los ficheros con la distribución por sexo y nivel de estudios

Se han estructurado los datos en la carpeta de inputs para la dimensión socioeconómica de forma que se dispone de un fichero CSV a nivel
de provincia. Así pues, se han definido una función en las utilidades que se encargan de cargar y dar una limpieza inicial a los datos.

Los datos se obtienen del censo anual de población:
https://www.ine.es/dynt3/inebase/index.htm?padre=10607&capsel=10609

In [2]:
path = os.path.join(DATA_INPUTS_DS, "Poblacion mayor 15 por sexo y estudios")

por_sexo_y_estudios = carga_datos_ine(path)

# Veo una muestra de su estructura y contenido
print(por_sexo_y_estudios.info())
por_sexo_y_estudios.sample(5)

<class 'pandas.core.frame.DataFrame'>
Index: 159480 entries, 90 to 261044
Data columns (total 8 columns):
 #   Column                        Non-Null Count   Dtype 
---  ------                        --------------   ----- 
 0   Provincias                    159480 non-null  object
 1   Municipios                    159480 non-null  object
 2   Secciones                     159480 non-null  object
 3   Sexo                          159480 non-null  object
 4   Nivel de formación alcanzado  159480 non-null  object
 5   Periodo                       159480 non-null  int64 
 6   Total                         145635 non-null  object
 7   Provincia                     159480 non-null  object
dtypes: int64(1), object(7)
memory usage: 11.0+ MB
None


,Provincias,Municipios,Secciones,Sexo,Nivel de formación alcanzado,Periodo,Total,Provincia
107008,34 Palencia,34109 Moratinos,3410901001 Moratinos sección 01001,Mujeres,Educación superior,2022,6,Palencia
195957,42 Soria,42173 Soria,4217302014 Soria sección 02014,Hombres,Educación superior,2023,404,Soria
195987,42 Soria,42173 Soria,4217303001 Soria sección 03001,Total,Educación superior,2023,425,Soria
92251,24 León,24173 Turcia,2417301001 Turcia sección 01001,Total,Total,2022,923,Leon
49793,09 Burgos,09219 Miranda de Ebro,0921903018 Miranda de Ebro sección 03018,Hombres,Primera etapa de Educación Secundaria y similar,2021,94,Burgos


In [3]:
hombres = por_sexo_y_estudios[
    (por_sexo_y_estudios["Sexo"] == "Hombres") &
    (por_sexo_y_estudios["Nivel de formación alcanzado"] != "Total")
].copy()

mujeres = por_sexo_y_estudios[
    (por_sexo_y_estudios["Sexo"] == "Mujeres") &
    (por_sexo_y_estudios["Nivel de formación alcanzado"] != "Total")
].copy()

hombres["Nivel de formación alcanzado"] = "Hombres " + hombres["Nivel de formación alcanzado"]
mujeres["Nivel de formación alcanzado"] = "Mujeres " + mujeres["Nivel de formación alcanzado"]


# Estandarización del dataframe de datos del INE

Como se puede observar, el fichero csv de datos del INE tiene un formato poco amigable para el tratamiento de los datos. En lugar de tener una fila
por cada par sección-año y varias columnas (una por factor), tiene múltiples filas con distintos indicadores para una misma sección, lo que resulta
complejo de tratar. Además, se observa como se mezcla el código del municipio, distrito y seccion con el texto, y deberían tener una columna con
los códigos.

In [4]:
hombres_estandarizado = estandarizar_df_ine(hombres, "Nivel de formación alcanzado")
print(hombres_estandarizado.info())
hombres_estandarizado.sample(5)

<class 'pandas.core.frame.DataFrame'>
Index: 42528 entries, 108 to 261029
Data columns (total 6 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   Provincia  42528 non-null  object
 1   CMuni      42528 non-null  object
 2   CUSEC      42528 non-null  object
 3   Indicador  42528 non-null  object
 4   Periodo    42528 non-null  int64 
 5   Total      38836 non-null  object
dtypes: int64(1), object(5)
memory usage: 2.3+ MB
None


,Provincia,CMuni,CUSEC,Indicador,Periodo,Total
99786,Palencia,34020,3402001001,Hombres Primera etapa de Educación Secundaria ...,2023,15
170706,Segovia,40129,4012901001,Hombres Primera etapa de Educación Secundaria ...,2023,34
78861,Leon,24089,2408904001,Hombres Primera etapa de Educación Secundaria ...,2023,112
163964,Segovia,40051,4005101001,Hombres Educación superior,2021,NaN
164903,Segovia,40061,4006101001,Hombres Primera etapa de Educación Secundaria ...,2021,13


In [5]:
mujeres_estandarizado = estandarizar_df_ine(mujeres, "Nivel de formación alcanzado")
print(mujeres_estandarizado.info())
mujeres_estandarizado.sample(5)

<class 'pandas.core.frame.DataFrame'>
Index: 42528 entries, 123 to 261044
Data columns (total 6 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   Provincia  42528 non-null  object
 1   CMuni      42528 non-null  object
 2   CUSEC      42528 non-null  object
 3   Indicador  42528 non-null  object
 4   Periodo    42528 non-null  int64 
 5   Total      38836 non-null  object
dtypes: int64(1), object(5)
memory usage: 2.3+ MB
None


,Provincia,CMuni,CUSEC,Indicador,Periodo,Total
238310,Zamora,49025,4902501001,Mujeres Educación primaria e inferior,2021,19
80358,Leon,24089,2408906008,Mujeres Educación primaria e inferior,2023,37
238398,Zamora,49026,4902601001,Mujeres Educación primaria e inferior,2023,42
56463,Burgos,09316,0931601001,Mujeres Educación primaria e inferior,2023,13
21453,Avila,05228,0522801001,Mujeres Educación primaria e inferior,2023,19


In [6]:
# Concatenamos ambos dfs ya que ahora no comparten columnas
por_sexo_y_estudios_estandarizado = pd.concat(
    [hombres_estandarizado, 
     mujeres_estandarizado
    ],
    ignore_index=True
)
print(por_sexo_y_estudios_estandarizado.info())
por_sexo_y_estudios_estandarizado.sample(5)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 85056 entries, 0 to 85055
Data columns (total 6 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   Provincia  85056 non-null  object
 1   CMuni      85056 non-null  object
 2   CUSEC      85056 non-null  object
 3   Indicador  85056 non-null  object
 4   Periodo    85056 non-null  int64 
 5   Total      77672 non-null  object
dtypes: int64(1), object(5)
memory usage: 3.9+ MB
None


,Provincia,CMuni,CUSEC,Indicador,Periodo,Total
56247,Leon,24115,2411501001,Mujeres Primera etapa de Educación Secundaria ...,2023,216
71876,Soria,42009,4200901001,Mujeres Segunda etapa de Educación Secundaria ...,2021,4
58865,Palencia,34009,3400901001,Mujeres Primera etapa de Educación Secundaria ...,2021,NaN
39104,Zamora,49021,4902103002,Hombres Segunda etapa de Educación Secundaria ...,2021,97
50862,Burgos,09224,0922401001,Mujeres Segunda etapa de Educación Secundaria ...,2023,12


## Filtrado de años
Revisando la documentación del INE, en el año 2021 se cambió radicalemente la metodología que define las secciones censales,
y en concreto en Castilla y León se aumentó el numero de censos de 2700 a unos 3500 apróximadamente. Es por ello que, si bien
se dispone de datos de años anteriores, sería complejo y peligroso fragmentar y proyectar los censos de años previos en la malla 
censal actual, por lo que se filtraran datos de años previos

In [7]:
# Reviso los indicadores disponibles
revisar_indicadores_disponibles(por_sexo_y_estudios_estandarizado)

# Filtro por los años 2021 - 2023.
por_sexo_y_estudios_recientes = por_sexo_y_estudios_estandarizado[
    por_sexo_y_estudios_estandarizado["Periodo"].isin([2021, 2022, 2023])
].copy()

📅 Años disponibles:
[2023 2022 2021]
------------------------------------------------------------
🧩 Indicadores demográficos disponibles:
  - Hombres Educación primaria e inferior
  - Hombres Primera etapa de Educación Secundaria y similar
  - Hombres Segunda etapa de Educación Secundaria y Educación Postsecundaria no Superior
  - Hombres Educación superior
  - Mujeres Educación primaria e inferior
  - Mujeres Primera etapa de Educación Secundaria y similar
  - Mujeres Segunda etapa de Educación Secundaria y Educación Postsecundaria no Superior
  - Mujeres Educación superior
------------------------------------------------------------


In [8]:
# Pivoto los indicadores para tener una columna por indicador y reducir las filas de la tabla
por_sexo_y_estudios_por_seccion = pivotar_indicadores(por_sexo_y_estudios_recientes)
print(por_sexo_y_estudios_por_seccion.info())
por_sexo_y_estudios_por_seccion.sample(5)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9709 entries, 0 to 9708
Data columns (total 12 columns):
 #   Column                                                                                Non-Null Count  Dtype 
---  ------                                                                                --------------  ----- 
 0   Provincia                                                                             9709 non-null   object
 1   CMuni                                                                                 9709 non-null   object
 2   CUSEC                                                                                 9709 non-null   object
 3   Periodo                                                                               9709 non-null   int64 
 4   Hombres_Educación_primaria_e_inferior                                                 9709 non-null   object
 5   Hombres_Educación_superior                                                            9709

,Provincia,CMuni,CUSEC,Periodo,Hombres_Educación_primaria_e_inferior,Hombres_Educación_superior,Hombres_Primera_etapa_de_Educación_Secundaria_y_similar,Hombres_Segunda_etapa_de_Educación_Secundaria_y_Educación_Postsecundaria_no_Superior,Mujeres_Educación_primaria_e_inferior,Mujeres_Educación_superior,Mujeres_Primera_etapa_de_Educación_Secundaria_y_similar,Mujeres_Segunda_etapa_de_Educación_Secundaria_y_Educación_Postsecundaria_no_Superior
5919,Salamanca,37362,3736201003,2021,39,250,169,183,57,298,139,149
7145,Valladolid,47003,4700301001,2023,33,37,52,29,28,29,28,17
2435,Leon,24014,2401401005,2021,82,36,134,52,123,57,107,45
2409,Leon,24010,2401001004,2022,98,142,310,182,180,214,252,172
2369,Leon,24006,2400601001,2021,49,67,112,50,74,43,91,42


In [10]:
# Creamos totales
# Creamos totales
cols_num = [
    "Hombres_Educación_primaria_e_inferior", "Mujeres_Educación_primaria_e_inferior",
    "Hombres_Educación_superior", "Mujeres_Educación_superior",
    "Hombres_Primera_etapa_de_Educación_Secundaria_y_similar", "Mujeres_Primera_etapa_de_Educación_Secundaria_y_similar",
    "Hombres_Segunda_etapa_de_Educación_Secundaria_y_Educación_Postsecundaria_no_Superior", "Mujeres_Segunda_etapa_de_Educación_Secundaria_y_Educación_Postsecundaria_no_Superior"
]

for c in cols_num:
    por_sexo_y_estudios_por_seccion[c] = pd.to_numeric(
        por_sexo_y_estudios_por_seccion[c], errors="coerce"
    )

por_sexo_y_estudios_por_seccion["Total_Educación_primaria_e_inferior"] = (
    por_sexo_y_estudios_por_seccion["Hombres_Educación_primaria_e_inferior"] +
    por_sexo_y_estudios_por_seccion["Mujeres_Educación_primaria_e_inferior"]
)

por_sexo_y_estudios_por_seccion["Total_Educación_superior"] = (
    por_sexo_y_estudios_por_seccion["Hombres_Educación_superior"] +
    por_sexo_y_estudios_por_seccion["Mujeres_Educación_superior"]
)

por_sexo_y_estudios_por_seccion["Total_Primera_etapa_de_Educación_Secundaria_y_similar"] = (
    por_sexo_y_estudios_por_seccion["Hombres_Primera_etapa_de_Educación_Secundaria_y_similar"] +
    por_sexo_y_estudios_por_seccion["Mujeres_Primera_etapa_de_Educación_Secundaria_y_similar"]
)

por_sexo_y_estudios_por_seccion["Total_Segunda_etapa_de_Educación_Secundaria_y_Educación_Postsecundaria_no_Superior"] = (
    por_sexo_y_estudios_por_seccion["Hombres_Segunda_etapa_de_Educación_Secundaria_y_Educación_Postsecundaria_no_Superior"] +
    por_sexo_y_estudios_por_seccion["Mujeres_Segunda_etapa_de_Educación_Secundaria_y_Educación_Postsecundaria_no_Superior"]
)

print(por_sexo_y_estudios_por_seccion.info())
por_sexo_y_estudios_por_seccion.sample(5)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9709 entries, 0 to 9708
Data columns (total 16 columns):
 #   Column                                                                                Non-Null Count  Dtype 
---  ------                                                                                --------------  ----- 
 0   Provincia                                                                             9709 non-null   object
 1   CMuni                                                                                 9709 non-null   object
 2   CUSEC                                                                                 9709 non-null   object
 3   Periodo                                                                               9709 non-null   int64 
 4   Hombres_Educación_primaria_e_inferior                                                 9709 non-null   int64 
 5   Hombres_Educación_superior                                                            9709

,Provincia,CMuni,CUSEC,Periodo,Hombres_Educación_primaria_e_inferior,Hombres_Educación_superior,Hombres_Primera_etapa_de_Educación_Secundaria_y_similar,Hombres_Segunda_etapa_de_Educación_Secundaria_y_Educación_Postsecundaria_no_Superior,Mujeres_Educación_primaria_e_inferior,Mujeres_Educación_superior,Mujeres_Primera_etapa_de_Educación_Secundaria_y_similar,Mujeres_Segunda_etapa_de_Educación_Secundaria_y_Educación_Postsecundaria_no_Superior,Total_Educación_primaria_e_inferior,Total_Educación_superior,Total_Primera_etapa_de_Educación_Secundaria_y_similar,Total_Segunda_etapa_de_Educación_Secundaria_y_Educación_Postsecundaria_no_Superior
6524,Segovia,40194,4019404003,2023,68,118,92,88,90,151,94,115,158,269,186,203
8115,Valladolid,47186,4718606016,2021,59,40,109,66,113,68,106,74,172,108,215,140
5014,Salamanca,37183,3718301001,2022,39,33,84,34,59,35,67,25,98,68,151,59
8080,Valladolid,47186,4718606001,2022,34,161,64,82,75,230,87,83,109,391,151,165
1406,Burgos,09059,0905909024,2022,58,146,157,122,80,164,157,114,138,310,314,236


# Export de los resultados

In [11]:
# Creamos la carpeta si no existe
os.makedirs(DATA_OUTPUTS_DS, exist_ok=True)

# Rutas de salida
ruta_seccion = os.path.join(DATA_OUTPUTS_DS, "estudios_por_sexo_por_seccion.csv")

# Guardar DataFrames
por_sexo_y_estudios_por_seccion.to_csv(
    ruta_seccion,
    index=False,
    encoding="utf-8-sig",
    sep=";",          # separador de columnas compatible con Excel español
    decimal=",",      # separador decimal europeo
    float_format="%.3f"
)

print(f"✅ Archivos guardados correctamente en: {DATA_OUTPUTS_DS}")

✅ Archivos guardados correctamente en: D:\MASTER EN CIENCIA DE DATOS\TFM\TrabajoFinal\ivst-tfm\data\outputs\DS_Dim_socioeconomica
